Load

In [9]:
import pandas as pd
import folium

stations = pd.read_csv("../data/processed/stations.csv")
types = pd.read_csv("../data/station_types.csv")
matchups = pd.read_csv("../data/processed/matchups.csv")

# Same-day matchups per station
same_day = (matchups[matchups["day_difference"] == 0]
            .groupby("station")["sample_id"].nunique()
            .rename("same_day_matchups"))

st = stations.merge(types, on="station", how="left").join(same_day, on="station")
st["same_day_matchups"] = st["same_day_matchups"].fillna(0).astype(int)

Map

In [10]:
COLORS = {"open": "blue", "creek": "red"}

m = folium.Map(location=[40.70, -73.93], zoom_start=11, tiles=None)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery", name="Satellite").add_to(m)
folium.TileLayer("OpenStreetMap", name="Streets").add_to(m)

for row in st.itertuples():
    folium.CircleMarker(
        location=[row.lat, row.lon], radius=6,
        color="yellow" if row.check == "check" else COLORS[row.type],   # yellow ring = to check
        weight=3, fill=True, fill_color=COLORS[row.type], fill_opacity=0.9,
        popup=folium.Popup(
            f"<b>{row.station}</b> ({row.waterway_name})<br>type: {row.type}"
            f"<br>samples: {row.n_observations}<br>same-day matchups: {row.same_day_matchups}",
            max_width=250),
        tooltip=row.station,
    ).add_to(m)

from folium.plugins import MeasureControl
m.add_child(MeasureControl(primary_length_unit="meters"))

folium.LayerControl().add_to(m)
m.save("../results/station_map.html")
m

Summary

In [3]:
st.groupby("type").agg(stations=("station", "size"),
                       samples=("n_observations", "sum"),
                       same_day_matchups=("same_day_matchups", "sum"))

,stations,samples,same_day_matchups
type,,,
creek,39,7743,1564
open,51,11236,2224
